# Whisper large-v3 Studio on Colab

Transcribe an audio or video file with [openai/whisper-large-v3](https://huggingface.co/openai/whisper-large-v3) on Colab's GPU and download timestamped subtitles. Code: [letsrunai/whisper-large-v3-studio](https://github.com/letsrunai/whisper-large-v3-studio) (v0.1.0).

1. **Runtime → Change runtime type → T4 GPU** (any GPU works).
2. Run the cells from top to bottom: install, upload, transcribe, download.

Everything below runs from the notebook cells, which is how Colab's free tier is meant to be used. The web app at the end is optional: Colab's free tier does not allow working mainly through a web UI, so run that part only with compute units (Colab Pro or pay as you go). See the [Colab FAQ](https://research.google.com/colaboratory/faq.html).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install "git+https://github.com/letsrunai/whisper-large-v3-studio@v0.1.0#subdirectory=server"

## 1. Upload a file

Any audio or video file ffmpeg can read: mp3, wav, m4a, mp4, webm…

In [ ]:
from google.colab import files

uploaded = files.upload()
audio = next(iter(uploaded))
print("Uploaded:", audio)

## 2. Transcribe

Leave the language empty to detect it, or set a code such as `en` or `tr`. Tick translate to get English text from any language.

The first run downloads the weights (about 3 GB) and loads them onto the GPU, so it takes longer than the next ones. The last lines show the time it took, the speed against real time and the peak GPU memory.

In [ ]:
LANGUAGE = ""  # @param {type:"string"}
TRANSLATE = False  # @param {type:"boolean"}

import subprocess

args = ["whisper-studio", "transcribe", audio, "--srt", "subtitles.srt"]
if LANGUAGE:
    args += ["--language", LANGUAGE]
if TRANSLATE:
    args.append("--translate")

monitor = subprocess.Popen(
    ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits", "-lms", "500"],
    stdout=subprocess.PIPE,
    text=True,
)
# Captured and printed here: a child process's own output does not reach the cell.
result = subprocess.run(args, capture_output=True, text=True)
monitor.terminate()
print(result.stdout)
print(result.stderr.strip().splitlines()[-1] if result.stderr.strip() else "")
if result.returncode:
    raise RuntimeError(result.stderr)
print("peak GPU memory:", max(int(line) for line in monitor.communicate()[0].split()), "MiB")

## 3. Download the subtitles

In [ ]:
print(open("subtitles.srt", encoding="utf-8").read()[:600])
files.download("subtitles.srt")

## Optional: the web app (needs compute units)

The same page you get on your own machine, opened inside Colab. **Colab's free tier does not allow working mainly through a web UI, and may end such sessions**; run this only on a runtime paid with compute units. On the free tier, use the cells above.

On your own machine you get the full Next.js page; see the [README](https://github.com/letsrunai/whisper-large-v3-studio#1-on-your-own-machine).

In [ ]:
import subprocess, time, urllib.request

server = subprocess.Popen(["whisper-studio", "serve", "--port", "7860"])
for _ in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:7860")
        break
    except OSError:
        time.sleep(1)

from google.colab import output

output.serve_kernel_port_as_window(7860)